# Phase 4: High-Fidelity Backtesting Engine & Performance Attribution

**Objectives:**
1. Apply realistic transaction cost model (commissions + market impact + borrow costs)
2. Compute comprehensive performance metrics (Gross & Net)
3. Fama-French 5-Factor attribution — isolate Alpha from systematic exposures
4. Rolling risk/return analysis with visualizations
5. Drawdown analysis & stress-test decomposition
6. Long vs Short leg attribution

---

**Inputs:**
| File | Description |
|---|---|
| `sp500_portfolio_weights.csv.gz` | Monthly optimal weights (Task 3.2) |
| `sp500_backtest_results.csv.gz` | Monthly gross returns & diagnostics |
| `sp500_monthly_returns.csv.gz` | Realized stock returns |
| `sp500_expected_returns.csv.gz` | Sector info for each stock |
| `F-F_Research_Data_5_Factors_2x3_daily.CSV` | FF5 factors |

In [3]:
# ============================================================
# Cell 1: Imports & Load Data
# ============================================================
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.dates import YearLocator, DateFormatter
import os
import warnings
warnings.filterwarnings("ignore")

# Publication-quality style
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

DATA_DIR = "data/"
FIG_DIR  = "figures/"
os.makedirs(FIG_DIR, exist_ok=True)

# --- Backtest results (from Task 3.2) ---
bt = pd.read_csv(DATA_DIR + "sp500_backtest_results.csv.gz", compression="gzip")
bt["month"] = pd.to_datetime(bt["month"]).dt.to_period("M")
bt["date"] = bt["month"].apply(lambda m: m.to_timestamp(how="end"))
print(f"Backtest results: {len(bt)} months")

# --- Portfolio weights ---
weights_df = pd.read_csv(DATA_DIR + "sp500_portfolio_weights.csv.gz",
                         compression="gzip")
weights_df["month"] = pd.to_datetime(weights_df["month"]).dt.to_period("M")
print(f"Portfolio weights: {len(weights_df):,} records")

# --- Monthly returns ---
monthly_ret = pd.read_csv(DATA_DIR + "sp500_monthly_returns.csv.gz",
                          compression="gzip")
monthly_ret["last_date"] = pd.to_datetime(monthly_ret["last_date"])
monthly_ret["month"] = monthly_ret["last_date"].dt.to_period("M")
print(f"Monthly returns: {monthly_ret.shape[0]:,} rows")

# --- Expected returns (for sector info) ---
er = pd.read_csv(DATA_DIR + "sp500_expected_returns.csv.gz", compression="gzip")
er["month"] = pd.to_datetime(er["date"]).dt.to_period("M")
print(f"Expected returns: {er.shape[0]:,} rows")

# --- Fama-French 5 factors ---
ff5_raw = pd.read_csv(DATA_DIR + "F-F_Research_Data_5_Factors_2x3_daily.CSV",
                      skiprows=3)
ff5_raw.columns = [c.strip() for c in ff5_raw.columns]
date_col = ff5_raw.columns[0]
ff5_raw = ff5_raw.rename(columns={date_col: "date_raw"})
ff5_raw = ff5_raw[ff5_raw["date_raw"].astype(str).str.match(r"^\d{8}$", na=False)].copy()
ff5_raw["date"] = pd.to_datetime(ff5_raw["date_raw"].astype(str))
for c in ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]:
    ff5_raw[c] = pd.to_numeric(ff5_raw[c], errors="coerce")
ff5_raw["month"] = ff5_raw["date"].dt.to_period("M")

ff5_monthly = ff5_raw.groupby("month").agg(
    mktrf=("Mkt-RF", lambda x: (1 + x/100).prod() - 1),
    smb=("SMB",      lambda x: (1 + x/100).prod() - 1),
    hml=("HML",      lambda x: (1 + x/100).prod() - 1),
    rmw=("RMW",      lambda x: (1 + x/100).prod() - 1),
    cma=("CMA",      lambda x: (1 + x/100).prod() - 1),
    rf=("RF",        lambda x: (1 + x/100).prod() - 1),
).reset_index()
print(f"FF5 monthly: {len(ff5_monthly)} months")

# Merge RF and FF5 onto backtest
# Use explicit suffixes to avoid rf column collision ambiguity
bt = bt.merge(ff5_monthly, on="month", how="left", suffixes=("", "_ff5"))

# Resolve risk-free column robustly:
# - if both left and right have rf -> right side is rf_ff5
# - if only right has rf -> column is rf
if "rf_ff5" in bt.columns:
    if "rf" in bt.columns:
        bt["rf"] = bt["rf_ff5"].fillna(bt["rf"])
    else:
        bt["rf"] = bt["rf_ff5"]
    bt.drop(columns=["rf_ff5"], inplace=True)

if "rf" not in bt.columns:
    raise KeyError("`rf` column missing after FF5 merge. Check source columns and merge keys.")

bt["excess_ret"] = bt["port_ret"] - bt["rf"]
print(f"\nBacktest period: {bt['month'].iloc[0]} → {bt['month'].iloc[-1]}")

Backtest results: 157 months
Portfolio weights: 32,405 records
Monthly returns: 140,070 rows
Expected returns: 99,363 rows
FF5 monthly: 738 months

Backtest period: 2011-12 → 2024-12


## Step 1: Transaction Cost Model

Three components:
1. **Fixed commission:** 5 bp one-way
2. **Market impact:** 5 bp one-way (simplified; S&P 500 stocks are highly liquid)
3. **Short-selling borrow cost:** 50 bp annualized on short exposure

Total one-way cost = 10 bp × turnover (two-way) + borrow cost.

In [4]:
# ============================================================
# Cell 2: Transaction Cost Model → Net Returns
# ============================================================

COMMISSION_BP   = 5     # one-way, basis points
IMPACT_BP       = 5     # one-way, basis points
BORROW_RATE_ANN = 0.0050  # 50 bp annualized

ONE_WAY_COST = (COMMISSION_BP + IMPACT_BP) / 10000  # 0.001

print("Transaction Cost Model:")
print(f"  Commission:    {COMMISSION_BP} bp one-way")
print(f"  Market impact: {IMPACT_BP} bp one-way")
print(f"  Total one-way: {ONE_WAY_COST*10000:.0f} bp")
print(f"  Borrow rate:   {BORROW_RATE_ANN:.2%} annualized")

# Trading cost: one-way cost × two-way turnover
bt["trading_cost"] = ONE_WAY_COST * bt["turnover"] * 2

# Borrow cost: monthly borrow on short leg
# Short exposure ≈ gross_leverage / 2 (since dollar neutral)
bt["short_exposure"] = bt["gross_leverage"] / 2
bt["borrow_cost"] = bt["short_exposure"] * BORROW_RATE_ANN / 12

# Total cost
bt["total_cost"] = bt["trading_cost"] + bt["borrow_cost"]

# Net returns
bt["net_ret"] = bt["port_ret"] - bt["total_cost"]
bt["net_excess_ret"] = bt["net_ret"] - bt["rf"]

# Net NAV
bt["gross_nav"] = (1 + bt["port_ret"].fillna(0)).cumprod()
bt["net_nav"]   = (1 + bt["net_ret"].fillna(0)).cumprod()

print(f"\n  Avg monthly costs:")
print(f"    Trading cost:  {bt['trading_cost'].mean():.4f} "
      f"({bt['trading_cost'].mean()*12:.2%} ann)")
print(f"    Borrow cost:   {bt['borrow_cost'].mean():.4f} "
      f"({bt['borrow_cost'].mean()*12:.2%} ann)")
print(f"    Total cost:    {bt['total_cost'].mean():.4f} "
      f"({bt['total_cost'].mean()*12:.2%} ann)")

Transaction Cost Model:
  Commission:    5 bp one-way
  Market impact: 5 bp one-way
  Total one-way: 10 bp
  Borrow rate:   0.50% annualized

  Avg monthly costs:
    Trading cost:  0.0011 (1.30% ann)
    Borrow cost:   0.0004 (0.50% ann)
    Total cost:    0.0015 (1.80% ann)


In [5]:
# ============================================================
# Cell 3: Comprehensive Performance Summary (Gross & Net)
# ============================================================

def compute_metrics(rets, excess_rets, nav_series, label=""):
    '''Compute standard performance metrics.'''
    T = len(rets.dropna())
    r = rets.dropna()
    ex = excess_rets.dropna()

    ann_ret = (1 + r).prod() ** (12 / T) - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe = ex.mean() / ex.std() * np.sqrt(12) if ex.std() > 0 else 0

    # Max drawdown
    cm = nav_series.cummax()
    dd = (nav_series - cm) / cm
    max_dd = dd.min()
    max_dd_idx = dd.idxmin()

    # Calmar
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 0 else np.nan

    # Longest drawdown (months to recover)
    underwater = dd < 0
    if underwater.any():
        groups = (~underwater).cumsum()
        dd_lengths = underwater.groupby(groups).sum()
        longest_dd = int(dd_lengths.max())
    else:
        longest_dd = 0

    return {
        "label": label,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_dd": max_dd,
        "calmar": calmar,
        "longest_dd_months": longest_dd,
        "win_rate": (r > 0).mean(),
        "final_nav": float(nav_series.iloc[-1]),
        "avg_monthly": r.mean(),
        "skew": r.skew(),
        "kurtosis": r.kurtosis(),
    }

gross_m = compute_metrics(bt["port_ret"], bt["excess_ret"], bt["gross_nav"], "Gross")
net_m   = compute_metrics(bt["net_ret"], bt["net_excess_ret"], bt["net_nav"], "Net")

print("=" * 70)
print("  PERFORMANCE SUMMARY")
print("=" * 70)
print(f"  Period: {bt['month'].iloc[0]} → {bt['month'].iloc[-1]} "
      f"({len(bt)} months)")

print(f"\n  {'Metric':<28s} {'Gross':>12s} {'Net':>12s}")
print("  " + "-" * 54)
rows = [
    ("Ann. Return",          f"{gross_m['ann_return']:+.2%}", f"{net_m['ann_return']:+.2%}"),
    ("Ann. Volatility",      f"{gross_m['ann_vol']:.2%}",     f"{net_m['ann_vol']:.2%}"),
    ("Sharpe Ratio",         f"{gross_m['sharpe']:.3f}",      f"{net_m['sharpe']:.3f}"),
    ("Max Drawdown",         f"{gross_m['max_dd']:.2%}",      f"{net_m['max_dd']:.2%}"),
    ("Calmar Ratio",         f"{gross_m['calmar']:.3f}",      f"{net_m['calmar']:.3f}"),
    ("Longest DD (months)",  f"{gross_m['longest_dd_months']}", f"{net_m['longest_dd_months']}"),
    ("Win Rate",             f"{gross_m['win_rate']:.1%}",    f"{net_m['win_rate']:.1%}"),
    ("Final NAV",            f"{gross_m['final_nav']:.4f}",   f"{net_m['final_nav']:.4f}"),
    ("Monthly Skew",         f"{gross_m['skew']:.3f}",        f"{net_m['skew']:.3f}"),
    ("Monthly Kurtosis",     f"{gross_m['kurtosis']:.3f}",    f"{net_m['kurtosis']:.3f}"),
]
for name, g, n in rows:
    print(f"  {name:<28s} {g:>12s} {n:>12s}")

  PERFORMANCE SUMMARY
  Period: 2011-12 → 2024-12 (157 months)

  Metric                              Gross          Net
  ------------------------------------------------------
  Ann. Return                       +11.81%       +9.83%
  Ann. Volatility                    18.21%       18.22%
  Sharpe Ratio                        0.637        0.538
  Max Drawdown                      -24.38%      -25.52%
  Calmar Ratio                        0.484        0.385
  Longest DD (months)                    22           23
  Win Rate                            61.5%        58.3%
  Final NAV                          4.2674       3.3839
  Monthly Skew                        0.126        0.132
  Monthly Kurtosis                    2.456        2.447


## Visualization: NAV Curve & Drawdown

In [6]:
# ============================================================
# Cell 4: NAV Curve & Drawdown Chart
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1],
                                sharex=True)

dates = bt["date"]

# --- NAV ---
ax1.plot(dates, bt["gross_nav"], color="#2563EB", linewidth=1.5,
         label=f'Gross (Sharpe={gross_m["sharpe"]:.2f})')
ax1.plot(dates, bt["net_nav"], color="#DC2626", linewidth=1.5,
         label=f'Net (Sharpe={net_m["sharpe"]:.2f})')
ax1.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax1.set_ylabel("NAV")
ax1.set_title("Market-Neutral Strategy: Cumulative Performance (2012–2024)",
              fontsize=14, fontweight="bold")
ax1.legend(loc="upper left", fontsize=11)
ax1.set_yscale("log")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}"))

# --- Drawdown ---
gross_dd = (bt["gross_nav"] - bt["gross_nav"].cummax()) / bt["gross_nav"].cummax()
net_dd   = (bt["net_nav"] - bt["net_nav"].cummax()) / bt["net_nav"].cummax()

ax2.fill_between(dates, gross_dd, 0, color="#2563EB", alpha=0.3, label="Gross")
ax2.fill_between(dates, net_dd, 0, color="#DC2626", alpha=0.3, label="Net")
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.legend(loc="lower left", fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR + "nav_drawdown.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}nav_drawdown.png")

  Saved: figures/nav_drawdown.png


In [7]:
# ============================================================
# Cell 5: Year-by-Year Performance (Gross & Net)
# ============================================================

bt["year"] = bt["month"].apply(lambda m: m.year)

print("=" * 78)
print("  YEAR-BY-YEAR PERFORMANCE")
print("=" * 78)
print(f"\n  {'Year':>6s} {'Gross':>8s} {'Net':>8s} {'Vol':>8s} "
      f"{'Sharpe_G':>9s} {'Sharpe_N':>9s} {'MaxDD':>8s} {'TO':>8s}")
print("  " + "-" * 70)

for year, yg in bt.groupby("year"):
    gr = yg["port_ret"].dropna()
    nr = yg["net_ret"].dropna()
    if len(gr) == 0:
        continue
    g_ret = (1 + gr).prod() - 1
    n_ret = (1 + nr).prod() - 1
    vol = gr.std() * np.sqrt(12)
    g_ex = yg["excess_ret"].dropna()
    n_ex = yg["net_excess_ret"].dropna()
    g_sh = g_ex.mean() / g_ex.std() * np.sqrt(12) if g_ex.std() > 0 else 0
    n_sh = n_ex.mean() / n_ex.std() * np.sqrt(12) if n_ex.std() > 0 else 0
    nav_y = (1 + gr).cumprod()
    mdd = ((nav_y - nav_y.cummax()) / nav_y.cummax()).min()
    to = yg["turnover"].mean()

    print(f"  {year:>6d} {g_ret:>+8.2%} {n_ret:>+8.2%} {vol:>8.2%} "
          f"{g_sh:>9.2f} {n_sh:>9.2f} {mdd:>8.2%} {to:>8.2%}")

bt.drop(columns=["year"], inplace=True)

  YEAR-BY-YEAR PERFORMANCE

    Year    Gross      Net      Vol  Sharpe_G  Sharpe_N    MaxDD       TO
  ----------------------------------------------------------------------
    2011   +3.21%   +2.97%     nan%      0.00      0.00    0.00%  100.00%
    2012  +10.09%   +8.17%   15.00%      0.71      0.59   -7.36%   52.61%
    2013  +29.31%  +27.09%   11.48%      2.31      2.15   -2.38%   52.61%
    2014  +15.59%  +13.59%   15.49%      1.01      0.90   -7.75%   52.96%
    2015   +4.31%   +2.41%   17.82%      0.32      0.22  -10.00%   56.40%
    2016  -13.25%  -14.92%   17.05%     -0.76     -0.88  -24.38%   59.05%
    2017  +19.93%  +17.89%   16.35%      1.15      1.04  -11.77%   51.85%
    2018  +19.34%  +17.49%   17.18%      1.01      0.92   -8.46%   45.06%
    2019  -11.15%  -12.65%   13.19%     -1.00     -1.12  -11.53%   48.85%
    2020  +26.36%  +23.66%   35.07%      0.82      0.76  -23.72%   70.02%
    2021   -1.62%   -3.21%   22.51%      0.03     -0.04  -12.70%   47.37%
    2022  +

In [8]:
# ============================================================
# Cell 6: Monthly Return Bar Chart
# ============================================================

fig, ax = plt.subplots(figsize=(14, 4))

colors = ["#22C55E" if r >= 0 else "#EF4444"
          for r in bt["net_ret"].fillna(0)]
ax.bar(bt["date"], bt["net_ret"].fillna(0), width=25, color=colors, alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Net Monthly Return")
ax.set_title("Monthly Net Returns", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

fig.tight_layout()
fig.savefig(FIG_DIR + "monthly_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}monthly_returns.png")

  Saved: figures/monthly_returns.png


## Fama-French 5-Factor Attribution

$$r_p(t) - r_f(t) = \alpha + \beta_1 \cdot \text{MktRF} + \beta_2 \cdot \text{SMB} + \beta_3 \cdot \text{HML} + \beta_4 \cdot \text{RMW} + \beta_5 \cdot \text{CMA} + \varepsilon$$

The intercept α measures **stock-selection skill** — returns unexplained by known risk factors.

In [10]:
# ============================================================
# Cell 7: Fama-French 5-Factor Regression
# ============================================================
try:
    import statsmodels.api as sm
except ImportError:
    raise ImportError(
        "statsmodels is required for FF5 attribution. "
        "Install with: pip install statsmodels"
    )

# --- Full-sample regression ---
factor_names = ["mktrf", "smb", "hml", "rmw", "cma"]
factor_labels = {"mktrf": "Mkt-RF", "smb": "SMB", "hml": "HML",
                 "rmw": "RMW", "cma": "CMA"}

valid = bt["excess_ret"].notna() & bt[factor_names].notna().all(axis=1)
Y = bt.loc[valid, "excess_ret"].values
X = bt.loc[valid, factor_names].values
X_const = sm.add_constant(X)

model = sm.OLS(Y, X_const).fit(cov_type="HC1")  # heteroscedasticity-robust

print("=" * 70)
print("  FAMA-FRENCH 5-FACTOR ATTRIBUTION (GROSS)")
print("=" * 70)
print(f"  Sample: {valid.sum()} months\n")

# Display results
coef_names = ["Alpha (monthly)"] + [factor_labels[f] for f in factor_names]
print(f"  {'Factor':<20s} {'Coef':>10s} {'Std Err':>10s} {'t-stat':>8s} "
      f"{'p-value':>9s}")
print("  " + "-" * 60)
for i, name in enumerate(coef_names):
    c = model.params[i]
    se = model.bse[i]
    t = model.tvalues[i]
    p = model.pvalues[i]
    sig = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))
    print(f"  {name:<20s} {c:>10.5f} {se:>10.5f} {t:>8.2f} {p:>9.4f} {sig}")

print(f"\n  Alpha (annualized): {model.params[0] * 12:>+.2%}")
print(f"  R-squared:          {model.rsquared:>.4f}")
print(f"  Adj R-squared:      {model.rsquared_adj:>.4f}")
print(f"  Durbin-Watson:      {sm.stats.stattools.durbin_watson(model.resid):.3f}")

# Interpretation
print(f"\n  Interpretation:")
if model.pvalues[0] < 0.05:
    print(f"  → Alpha is statistically significant at 5% level.")
    print(f"    Excess returns are NOT explained by FF5 systematic factors.")
else:
    print(f"  → Alpha is not significant at 5% (p={model.pvalues[0]:.3f}).")
    print(f"    Cannot reject that returns come from systematic factor exposure.")

mkt_beta = model.params[1]
print(f"  → Market beta = {mkt_beta:+.4f} "
      f"({'near zero — market neutral holds' if abs(mkt_beta) < 0.1 else 'non-trivial market exposure'})")

# --- Net returns regression ---
Y_net = bt.loc[valid, "net_excess_ret"].values
model_net = sm.OLS(Y_net, X_const).fit(cov_type="HC1")

print(f"\n  Net Alpha (monthly):    {model_net.params[0]:+.5f} "
      f"(t={model_net.tvalues[0]:.2f}, p={model_net.pvalues[0]:.4f})")
print(f"  Net Alpha (annualized): {model_net.params[0]*12:+.2%}")

  FAMA-FRENCH 5-FACTOR ATTRIBUTION (GROSS)
  Sample: 156 months

  Factor                     Coef    Std Err   t-stat   p-value
  ------------------------------------------------------------
  Alpha (monthly)         0.00843    0.00474     1.78    0.0755 *
  Mkt-RF                  0.15361    0.11567     1.33    0.1842 
  SMB                    -0.61308    0.22833    -2.69    0.0073 ***
  HML                     0.26962    0.22248     1.21    0.2256 
  RMW                    -0.49024    0.20839    -2.35    0.0186 **
  CMA                    -0.38528    0.32869    -1.17    0.2411 

  Alpha (annualized): +10.11%
  R-squared:          0.0781
  Adj R-squared:      0.0473
  Durbin-Watson:      2.179

  Interpretation:
  → Alpha is not significant at 5% (p=0.076).
    Cannot reject that returns come from systematic factor exposure.
  → Market beta = +0.1536 (non-trivial market exposure)

  Net Alpha (monthly):    +0.00694 (t=1.46, p=0.1432)
  Net Alpha (annualized): +8.33%


In [11]:
# ============================================================
# Cell 8: Rolling 36-Month Alpha
# ============================================================

ROLL_WINDOW = 36

rolling_alpha = []
rolling_t = []
rolling_dates = []

valid_idx = bt.index[valid]

for end in range(ROLL_WINDOW, len(valid_idx) + 1):
    start = end - ROLL_WINDOW
    idx_slice = valid_idx[start:end]

    Y_r = bt.loc[idx_slice, "excess_ret"].values
    X_r = sm.add_constant(bt.loc[idx_slice, factor_names].values)

    try:
        m = sm.OLS(Y_r, X_r).fit()
        rolling_alpha.append(m.params[0] * 12)   # annualized
        rolling_t.append(m.tvalues[0])
        rolling_dates.append(bt.loc[idx_slice[-1], "date"])
    except Exception:
        continue

rolling_df = pd.DataFrame({
    "date": rolling_dates,
    "alpha_ann": rolling_alpha,
    "t_stat": rolling_t,
})

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Rolling alpha
ax1.plot(rolling_df["date"], rolling_df["alpha_ann"],
         color="#2563EB", linewidth=1.5)
ax1.axhline(0, color="black", linewidth=0.8)
ax1.fill_between(rolling_df["date"], rolling_df["alpha_ann"], 0,
                 where=rolling_df["alpha_ann"] >= 0, color="#22C55E", alpha=0.2)
ax1.fill_between(rolling_df["date"], rolling_df["alpha_ann"], 0,
                 where=rolling_df["alpha_ann"] < 0, color="#EF4444", alpha=0.2)
ax1.set_ylabel("Annualized Alpha")
ax1.set_title("Rolling 36-Month FF5 Alpha", fontsize=13, fontweight="bold")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Rolling t-stat
ax2.plot(rolling_df["date"], rolling_df["t_stat"],
         color="#7C3AED", linewidth=1.5)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.axhline(1.96, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax2.axhline(-1.96, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax2.set_ylabel("t-statistic")
ax2.set_xlabel("")

fig.tight_layout()
fig.savefig(FIG_DIR + "rolling_alpha.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}rolling_alpha.png")

# Summary
print(f"\n  Rolling Alpha Summary:")
print(f"    Mean:      {rolling_df['alpha_ann'].mean():+.2%}")
print(f"    % > 0:     {(rolling_df['alpha_ann'] > 0).mean():.1%}")
print(f"    % t > 1.96:{(rolling_df['t_stat'] > 1.96).mean():.1%}")

  Saved: figures/rolling_alpha.png

  Rolling Alpha Summary:
    Mean:      +9.32%
    % > 0:     85.1%
    % t > 1.96:4.1%


In [12]:
# ============================================================
# Cell 9: Factor Exposure Analysis
# ============================================================

# Rolling 12-month beta to each FF5 factor
ROLL_BETA = 12

beta_records = []
for end in range(ROLL_BETA, len(valid_idx) + 1):
    start = end - ROLL_BETA
    idx_s = valid_idx[start:end]
    Y_b = bt.loc[idx_s, "excess_ret"].values
    X_b = sm.add_constant(bt.loc[idx_s, factor_names].values)
    try:
        m = sm.OLS(Y_b, X_b).fit()
        rec = {"date": bt.loc[idx_s[-1], "date"]}
        for j, fn in enumerate(factor_names):
            rec[fn] = m.params[j + 1]
        beta_records.append(rec)
    except Exception:
        continue

beta_df = pd.DataFrame(beta_records)

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()

for i, fn in enumerate(factor_names):
    ax = axes[i]
    ax.plot(beta_df["date"], beta_df[fn], color="#2563EB", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Rolling 12m Beta: {factor_labels[fn]}", fontsize=11)
    ax.set_ylabel("Beta")

# Hide extra subplot
axes[5].set_visible(False)

fig.suptitle("Rolling Factor Exposures", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR + "factor_exposures.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}factor_exposures.png")

# Summary table
print(f"\n  Average Factor Exposures (rolling 12m):")
print(f"  {'Factor':<12s} {'Mean β':>8s} {'Std β':>8s} {'Mean |β|':>9s}")
print("  " + "-" * 40)
for fn in factor_names:
    s = beta_df[fn]
    print(f"  {factor_labels[fn]:<12s} {s.mean():>+8.4f} {s.std():>8.4f} "
          f"{s.abs().mean():>9.4f}")

  Saved: figures/factor_exposures.png

  Average Factor Exposures (rolling 12m):
  Factor         Mean β    Std β  Mean |β|
  ----------------------------------------
  Mkt-RF        +0.1382   0.6080    0.5116
  SMB           -0.8017   0.8527    0.9522
  HML           +0.2119   1.1799    0.9843
  RMW           -0.5900   1.1324    0.9910
  CMA           -0.3189   1.7083    1.4462


## Drawdown Analysis & Stress Periods

In [13]:
# ============================================================
# Cell 10: Drawdown Analysis
# ============================================================

# Identify top 5 drawdown periods
gross_nav = bt["gross_nav"].values
dates_arr = bt["date"].values

cummax_arr = np.maximum.accumulate(gross_nav)
dd_arr = (gross_nav - cummax_arr) / cummax_arr

# Find drawdown troughs
print("=" * 70)
print("  TOP DRAWDOWN PERIODS")
print("=" * 70)

# Simple approach: find the 5 deepest points
dd_series = pd.Series(dd_arr, index=bt.index)
worst_months = dd_series.nsmallest(5)

print(f"\n  {'Rank':<6s} {'Trough Month':<15s} {'Depth':>8s} {'NAV':>8s}")
print("  " + "-" * 40)
for rank, (idx, depth) in enumerate(worst_months.items(), 1):
    m = bt.loc[idx, "month"]
    nav_val = bt.loc[idx, "gross_nav"]
    print(f"  {rank:<6d} {str(m):<15s} {depth:>8.2%} {nav_val:>8.4f}")

# Detailed analysis of worst drawdown
worst_idx = worst_months.index[0]
worst_month = bt.loc[worst_idx, "month"]

# Find start of this drawdown (last peak before trough)
peak_idx = worst_idx
while peak_idx > 0 and dd_arr[peak_idx - 1] < 0:
    peak_idx -= 1
peak_month = bt.loc[peak_idx, "month"]

# Find end (recovery to peak)
recovery_idx = worst_idx
while recovery_idx < len(bt) - 1 and dd_arr[recovery_idx] < 0:
    recovery_idx += 1
recovery_month = bt.loc[recovery_idx, "month"] if dd_arr[recovery_idx] >= 0 \
    else "Not recovered"

print(f"\n  Worst Drawdown Detail:")
print(f"    Peak:     {peak_month}")
print(f"    Trough:   {worst_month}")
print(f"    Recovery: {recovery_month}")
print(f"    Depth:    {dd_arr[worst_idx]:.2%}")
print(f"    Duration: {worst_idx - peak_idx} months (peak to trough)")

# Monthly returns during the drawdown
dd_slice = bt.iloc[peak_idx:worst_idx+1]
print(f"\n    Monthly returns during drawdown:")
for _, row in dd_slice.iterrows():
    print(f"      {row['month']}: {row['port_ret']:+.4f}")

  TOP DRAWDOWN PERIODS

  Rank   Trough Month       Depth      NAV
  ----------------------------------------
  1      2016-09          -24.38%   1.4009
  2      2017-04          -23.86%   1.4105
  3      2020-11          -23.72%   2.0069
  4      2017-05          -21.31%   1.4579
  5      2017-06          -20.73%   1.4686

  Worst Drawdown Detail:
    Peak:     2016-03
    Trough:   2016-09
    Recovery: 2018-01
    Depth:    -24.38%
    Duration: 6 months (peak to trough)

    Monthly returns during drawdown:
      2016-03: -0.0850
      2016-04: -0.0022
      2016-05: -0.0172
      2016-06: -0.0629
      2016-07: +0.0060
      2016-08: -0.0312
      2016-09: -0.0773


## Long vs Short Leg Attribution

In [14]:
# ============================================================
# Cell 11: Long vs Short Leg Decomposition
# ============================================================

# For each month, decompose portfolio return into long and short contributions
ret_lookup = monthly_ret.set_index(["permno", "month"])["ret_monthly"]

long_rets = []
short_rets = []

for _, wg in weights_df.groupby("month"):
    month = wg["month"].iloc[0]
    next_month = month + 1

    long_ret = 0
    short_ret = 0

    for _, row in wg.iterrows():
        p = row["permno"]
        w = row["weight"]
        r = ret_lookup.get((p, next_month), np.nan)
        if np.isnan(r):
            continue
        if w > 0:
            long_ret += w * r
        else:
            short_ret += w * r  # w is negative, r*w captures short P&L

    long_rets.append({"month": month, "long_ret": long_ret, "short_ret": short_ret})

legs = pd.DataFrame(long_rets)
legs = legs.merge(bt[["month", "port_ret"]], on="month", how="inner")

print("=" * 70)
print("  LONG vs SHORT LEG ATTRIBUTION")
print("=" * 70)

# Summary
for leg, col in [("Long leg", "long_ret"), ("Short leg", "short_ret")]:
    s = legs[col].dropna()
    ann = s.mean() * 12
    print(f"\n  {leg}:")
    print(f"    Avg monthly: {s.mean():+.4f} ({ann:+.2%} ann)")
    print(f"    Std monthly: {s.std():.4f}")
    print(f"    % positive:  {(s > 0).mean():.1%}")

# Correlation between legs
corr = legs["long_ret"].corr(legs["short_ret"])
print(f"\n  Correlation between legs: {corr:+.3f}")
print(f"  (Negative = good hedging; positive = legs move together)")

# Chart
fig, ax = plt.subplots(figsize=(14, 5))
long_cum = (1 + legs["long_ret"]).cumprod()
short_cum = (1 + legs["short_ret"]).cumprod()

dates_legs = legs["month"].apply(lambda m: m.to_timestamp(how="end"))
ax.plot(dates_legs, long_cum, color="#22C55E", linewidth=1.5, label="Long leg")
ax.plot(dates_legs, short_cum, color="#EF4444", linewidth=1.5, label="Short leg")
ax.plot(dates_legs, (1 + legs["port_ret"]).cumprod(),
        color="#2563EB", linewidth=2, label="Combined")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_ylabel("Cumulative Return (1 = starting)")
ax.set_title("Long vs Short Leg Cumulative Performance",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)

fig.tight_layout()
fig.savefig(FIG_DIR + "long_short_legs.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}long_short_legs.png")

  LONG vs SHORT LEG ATTRIBUTION

  Long leg:
    Avg monthly: +0.0220 (+26.39% ann)
    Std monthly: 0.0772
    % positive:  65.0%

  Short leg:
    Avg monthly: -0.0114 (-13.62% ann)
    Std monthly: 0.0745
    % positive:  42.7%

  Correlation between legs: -0.762
  (Negative = good hedging; positive = legs move together)
  Saved: figures/long_short_legs.png


In [15]:
# ============================================================
# Cell 12: Comparison with Market (SPY proxy)
# ============================================================

# Use Mkt-RF + RF as SPY proxy
bt["spy_ret"] = bt["mktrf"] + bt["rf"]
bt["spy_nav"] = (1 + bt["spy_ret"].fillna(0)).cumprod()

spy_m = compute_metrics(bt["spy_ret"], bt["mktrf"], bt["spy_nav"], "SPY")

print("=" * 70)
print("  STRATEGY vs MARKET COMPARISON")
print("=" * 70)

print(f"\n  {'Metric':<28s} {'Strategy(Net)':>14s} {'Market(SPY)':>12s}")
print("  " + "-" * 56)
for name, nk, sk in [
    ("Ann. Return",     f"{net_m['ann_return']:+.2%}",  f"{spy_m['ann_return']:+.2%}"),
    ("Ann. Volatility", f"{net_m['ann_vol']:.2%}",      f"{spy_m['ann_vol']:.2%}"),
    ("Sharpe Ratio",    f"{net_m['sharpe']:.3f}",       f"{spy_m['sharpe']:.3f}"),
    ("Max Drawdown",    f"{net_m['max_dd']:.2%}",       f"{spy_m['max_dd']:.2%}"),
    ("Final NAV",       f"{net_m['final_nav']:.4f}",    f"{spy_m['final_nav']:.4f}"),
]:
    print(f"  {name:<28s} {nk:>14s} {sk:>12s}")

# Correlation
corr_spy = bt["net_ret"].corr(bt["spy_ret"])
print(f"\n  Correlation with market: {corr_spy:+.3f}")
print(f"  {'→ Market neutral holds' if abs(corr_spy) < 0.3 else '→ Some market correlation'}")

# Chart
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(bt["date"], bt["net_nav"], color="#2563EB", linewidth=2,
        label=f'Strategy Net (Sharpe={net_m["sharpe"]:.2f})')
ax.plot(bt["date"], bt["spy_nav"], color="#9CA3AF", linewidth=1.5,
        label=f'SPY (Sharpe={spy_m["sharpe"]:.2f})')
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_ylabel("NAV (log scale)")
ax.set_title("Strategy vs Market", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}"))

fig.tight_layout()
fig.savefig(FIG_DIR + "strategy_vs_spy.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}strategy_vs_spy.png")

  STRATEGY vs MARKET COMPARISON

  Metric                        Strategy(Net)  Market(SPY)
  --------------------------------------------------------
  Ann. Return                          +9.83%      +14.50%
  Ann. Volatility                      18.22%       14.64%
  Sharpe Ratio                          0.538        0.915
  Max Drawdown                        -25.52%      -25.05%
  Final NAV                            3.3839       5.8778

  Correlation with market: +0.061
  → Market neutral holds
  Saved: figures/strategy_vs_spy.png


In [16]:
# ============================================================
# Cell 13: Portfolio Characteristics Over Time
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

# Turnover
ax = axes[0, 0]
ax.bar(bt["date"], bt["turnover"], width=25, color="#6366F1", alpha=0.7)
ax.set_ylabel("One-way Turnover")
ax.set_title("Monthly Turnover")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Gross Leverage
ax = axes[0, 1]
ax.plot(bt["date"], bt["gross_leverage"], color="#2563EB", linewidth=1.2)
ax.set_ylabel("Gross Leverage")
ax.set_title("Gross Leverage")

# Number of positions
ax = axes[1, 0]
ax.plot(bt["date"], bt["n_long"], color="#22C55E", linewidth=1.2, label="Long")
ax.plot(bt["date"], bt["n_short"], color="#EF4444", linewidth=1.2, label="Short")
ax.set_ylabel("# Positions")
ax.set_title("Number of Positions")
ax.legend(fontsize=10)

# Predicted vol
ax = axes[1, 1]
ax.plot(bt["date"], bt["port_vol_ann"], color="#F59E0B", linewidth=1.2)
ax.axhline(0.10, color="gray", linestyle="--", linewidth=0.8, alpha=0.5,
           label="10% target")
ax.set_ylabel("Ann. Vol")
ax.set_title("Predicted Portfolio Volatility")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=10)

fig.suptitle("Portfolio Characteristics Over Time",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR + "portfolio_characteristics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}portfolio_characteristics.png")

  Saved: figures/portfolio_characteristics.png


In [17]:
# ============================================================
# Cell 14: Save Final Outputs
# ============================================================

# --- Save enhanced backtest results ---
bt_out = bt.copy()
bt_out["month"] = bt_out["month"].astype(str)

# Drop intermediate columns that can't serialize
for col in bt_out.columns:
    if bt_out[col].dtype == "object" and col != "month" and col != "constraint_tag" \
            and col != "solver_status":
        try:
            bt_out[col] = bt_out[col].astype(str)
        except Exception:
            bt_out.drop(columns=[col], inplace=True)

bt_path = DATA_DIR + "sp500_phase4_full_report.csv.gz"
save_cols = ["month", "port_ret", "net_ret", "excess_ret", "net_excess_ret",
             "gross_nav", "net_nav", "trading_cost", "borrow_cost", "total_cost",
             "gross_leverage", "net_exposure", "turnover", "n_long", "n_short",
             "port_vol_ann", "max_sector_exp", "constraint_tag", "solver_status"]
save_cols = [c for c in save_cols if c in bt_out.columns]
bt_out[save_cols].to_csv(bt_path, index=False, compression="gzip")
bt_size = os.path.getsize(bt_path) / 1e6

print("=" * 70)
print("  PHASE 4 OUTPUTS SAVED")
print("=" * 70)

print(f"\n  1) {bt_path} ({bt_size:.2f} MB)")
print(f"     Complete monthly backtest with gross & net returns")

print(f"\n  2) Figures saved in {FIG_DIR}:")
for fig_name in sorted(os.listdir(FIG_DIR)):
    fsize = os.path.getsize(os.path.join(FIG_DIR, fig_name)) / 1e6
    print(f"     {fig_name} ({fsize:.2f} MB)")

# Final summary
print(f"\n{'=' * 70}")
print(f"  PROJECT COMPLETE — FINAL SUMMARY")
print(f"{'=' * 70}")
print(f"\n  Strategy: Multi-Factor Market-Neutral (Value + Size + Momentum)")
print(f"  Universe: S&P 500 constituents")
print(f"  Period:   {bt['month'].iloc[0]} → {bt['month'].iloc[-1]}")
print(f"\n  Gross Performance:")
print(f"    Return: {gross_m['ann_return']:+.2%}  |  Vol: {gross_m['ann_vol']:.2%}"
      f"  |  Sharpe: {gross_m['sharpe']:.3f}  |  MaxDD: {gross_m['max_dd']:.2%}")
print(f"\n  Net Performance (after 10bp one-way + 50bp borrow):")
print(f"    Return: {net_m['ann_return']:+.2%}  |  Vol: {net_m['ann_vol']:.2%}"
      f"  |  Sharpe: {net_m['sharpe']:.3f}  |  MaxDD: {net_m['max_dd']:.2%}")
print(f"\n  FF5 Alpha (annualized): {model.params[0]*12:+.2%} "
      f"(t={model.tvalues[0]:.2f}, p={model.pvalues[0]:.4f})")
print(f"  Market Beta:            {model.params[1]:+.4f}")
print(f"  Market Correlation:     {corr_spy:+.3f}")
print(f"\n  Phase 4 COMPLETE")
print(f"  → All deliverables ready for final report and presentation")

  PHASE 4 OUTPUTS SAVED

  1) data/sp500_phase4_full_report.csv.gz (0.02 MB)
     Complete monthly backtest with gross & net returns

  2) Figures saved in figures/:
     factor_exposures.png (0.28 MB)
     long_short_legs.png (0.10 MB)
     monthly_returns.png (0.03 MB)
     nav_drawdown.png (0.23 MB)
     portfolio_characteristics.png (0.22 MB)
     rolling_alpha.png (0.15 MB)
     strategy_vs_spy.png (0.13 MB)

  PROJECT COMPLETE — FINAL SUMMARY

  Strategy: Multi-Factor Market-Neutral (Value + Size + Momentum)
  Universe: S&P 500 constituents
  Period:   2011-12 → 2024-12

  Gross Performance:
    Return: +11.81%  |  Vol: 18.21%  |  Sharpe: 0.637  |  MaxDD: -24.38%

  Net Performance (after 10bp one-way + 50bp borrow):
    Return: +9.83%  |  Vol: 18.22%  |  Sharpe: 0.538  |  MaxDD: -25.52%

  FF5 Alpha (annualized): +10.11% (t=1.78, p=0.0755)
  Market Beta:            +0.1536
  Market Correlation:     +0.061

  Phase 4 COMPLETE
  → All deliverables ready for final report and presen